# ch04 Self-Attention 全手写（练习版）

> 补全 `____`，跑通所有 assert。真值全部来自实跑。
>
> 本章要写六个零件，每个都有对账对象 —— **对账差值是检验你写没写对的唯一标准**。

| 块 | 内容 | 难度 |
|---|---|---|
| T1 | 两版 softmax | ★ |
| T2 | 极端值验证 | ★ |
| T3 | QKV 投影 | ★ |
| T4 | 缩放点积注意力 | ★★★ |
| T5 | 与 `F.sdpa` 对账 | ★ |
| T6 | `d_k` 扫描 | ★★ |
| T7 | padding mask | ★★ |
| T8 | causal mask | ★★ |
| T9 | 多头 split-merge | ★★★ |
| T10 | 与 `nn.MultiheadAttention` 对账 | ★★ |
| T11 | 复杂度实测 | ★ |
| T12 | 注意力矩阵元素数 | ★ |
| T13 | 置换等变性 | ★★ |
| T14 | 真实句子的注意力矩阵 | ★ |

In [ ]:
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_num_threads(4)
torch.set_printoptions(precision=6, sci_mode=False)
print("torch", torch.__version__, "| 线程", torch.get_num_threads())

In [ ]:
# 脚手架：常量与工具函数两版都有，你只填 @@todo 块里的内容
B, T, E = 2, 4, 8                      # batch / 序列长度 / 模型维度（对账用的小形状）
E_MHA, H = 8, 2                        # 多头实验：维度 8，2 个头
D_HEAD = E_MHA // H


def R(t):
    """取标量值（顺手 detach，避免 requires_grad 警告）。"""
    return float(t.detach())


def near(a, b, eps=1e-6):
    return abs(float(a) - float(b)) < eps


print("B,T,E", B, T, E, "| E_MHA,H", E_MHA, H, "| d_head", D_HEAD)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

# macOS 上 matplotlib 默认字体不含中文，不设置的话图上全是方块
plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Arial Unicode MS", "STHeiti"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib", matplotlib.__version__, "| 中文字体 PingFang SC")

## T1 两版 softmax

In [ ]:
def softmax_naive(z):
    """朴素实现：先 exp 再归一化。"""

    e = torch.exp(z)
    return e / e.sum(-1, keepdim=True)


def softmax_stable(z):
    """数值稳定实现：先减每行最大值再 exp。"""

    e = torch.exp(z - z.max(-1, keepdim=True).values)
    return e / e.sum(-1, keepdim=True)


# ---- 验收 ----
probe_z = torch.randn(3, 5, generator=torch.Generator().manual_seed(1))
diff_softmax = R((softmax_stable(probe_z) - F.softmax(probe_z, -1)).abs().max())
assert diff_softmax < 1e-7, f"稳定版应与 F.softmax 一致，实际差 {diff_softmax}"
assert (softmax_naive(probe_z) - F.softmax(probe_z, -1)).abs().max() < 1e-7, \
    "小数值下朴素版也对 —— 问题只在极端值上暴露"
print("小输入下两版都对（与 F.softmax 差 %.1e，float32 舍入量级）" % diff_softmax)

## T2 极端值验证

In [ ]:
big = torch.tensor([[1000.0, 1000.0, 1000.0]])
big2 = torch.tensor([[1000.0, 999.0, 998.0]])

e_big = torch.exp(big)
naive_big = softmax_naive(big)
stable_big = softmax_stable(big)

print("exp([1000,1000,1000]) =", e_big.tolist(), "| 含 inf:", bool(torch.isinf(e_big).any()))
print("朴素 softmax ->", naive_big.tolist(), "| 含 nan:", bool(torch.isnan(naive_big).any()))
print("稳定 softmax ->", stable_big.tolist())
print()
print("差值版（最大值减到 0）:",
      [round(v, 6) for v in torch.exp(big2 - big2.max(-1, keepdim=True).values).tolist()[0]])
print("稳定 softmax:", [round(v, 6) for v in softmax_stable(big2).tolist()[0]])
print("手算校验 exp(0)/(1+e^-1+e^-2) = %.6f" % (1 / (1 + math.exp(-1) + math.exp(-2))))

print()
print("z       朴素版含 nan   稳定版")
for scale in (10.0, 100.0, 1000.0):
    z = torch.full((1, 3), scale)
    print("  %6.1f     %-12s   %.6f"
          % (scale, bool(torch.isnan(softmax_naive(z)).any()), R(softmax_stable(z).mean())))

# ---- 验收 ----
assert torch.isnan(naive_big).any(), "朴素版应当溢出成 nan"
assert not torch.isnan(stable_big).any()
assert near(R(stable_big[0, 0]), 1 / 3, 1e-6)
assert torch.isnan(softmax_naive(torch.full((1, 3), 100.0))).any(), "z=100 就该崩了"
print("\n结论：朴素版在 z≈100 就出 nan；减 max 是零成本的保险。")
print("注：因为 softmax(z) = softmax(z - c)，减常数不改变结果 —— 这不是近似。")

## T3 QKV 投影

In [ ]:
x = torch.randn(B, T, E, generator=torch.Generator().manual_seed(1))

with torch.random.fork_rng():                 # 权重初始化与外部 RNG 解耦
    torch.manual_seed(11)
    qkv_lin = nn.Linear(E, 3 * E)
    out_lin = nn.Linear(E, E)

W_qkv, b_qkv = qkv_lin.weight.detach(), qkv_lin.bias.detach()
print("x", tuple(x.shape), "| W_qkv", tuple(W_qkv.shape), "| b_qkv", tuple(b_qkv.shape))

In [ ]:
Q = x @ W_qkv[:E].T + b_qkv[:E]
K = x @ W_qkv[E:2 * E].T + b_qkv[E:2 * E]
V = x @ W_qkv[2 * E:].T + b_qkv[2 * E:]

# ---- 验收 ----
assert Q.shape == K.shape == V.shape == (B, T, E), "QKV 形状应与输入一致"
q_chunk, k_chunk, v_chunk = (x @ W_qkv.T + b_qkv).chunk(3, dim=-1)
assert torch.allclose(Q, q_chunk) and torch.allclose(V, v_chunk), "两种切法结果应当等价"
print("Q/K/V 形状", tuple(Q.shape))
print("Q[0,0,:3]", [round(v, 6) for v in Q[0, 0, :3].tolist()])
print("整块投影再 chunk 的结果完全一致：", torch.allclose(Q, q_chunk))

## T4 缩放点积注意力

In [ ]:
def attention(Q, K, V, mask=None):
    """返回 (输出 [.., T, d_v], 注意力权重 [.., T, T])"""

    d_k = Q.shape[-1]
    s = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    s = s.masked_fill(mask, float("-inf")) if mask is not None else s
    w = softmax_stable(s)
    return w @ V, w


# ---- 验收 ----
out_my, attn_my = attention(Q, K, V)
assert out_my.shape == (B, T, E), f"输出形状应为 {(B, T, E)}，实际 {tuple(out_my.shape)}"
assert attn_my.shape == (B, T, T)
assert (attn_my.sum(-1) - 1).abs().max() < 1e-6, "每一行的权重和必须是 1"
print("输出形状", tuple(out_my.shape), "| 权重形状", tuple(attn_my.shape))
print("每行权重和 %.6f" % R(attn_my.sum(-1).mean()))

## T5 与 `F.scaled_dot_product_attention` 对账

In [ ]:
out_sdpa = F.scaled_dot_product_attention(Q, K, V)

diff_sdpa = R((out_my - out_sdpa).abs().max())

# ---- 不除 sqrt(d_k) 会怎样 ----
raw_scores = Q @ K.transpose(-2, -1)
scaled_scores = raw_scores / math.sqrt(E)
w_noscale = softmax_stable(raw_scores)
out_noscale = w_noscale @ V

print("手写 vs F.sdpa 最大绝对差: %.3e" % diff_sdpa)
print("不除 sqrt(d_k) 的输出差异: %.6f" % R((out_noscale - out_my).abs().max()))
print("缩放前 scores 标准差 %.6f -> 缩放后 %.6f"
      % (R(raw_scores.std()), R(scaled_scores.std())))
print("缩放前权重熵 %.6f -> 缩放后 %.6f"
      % (R(-(w_noscale * torch.log(w_noscale + 1e-12)).sum(-1).mean()),
         R(-(attn_my * torch.log(attn_my + 1e-12)).sum(-1).mean())))

# ---- 验收 ----
assert diff_sdpa < 1e-6, f"手写注意力与 F.sdpa 不一致：{diff_sdpa}"
assert R(w_noscale.std()) > R(attn_my.std()), "不缩放时权重应当更极端"
print("\n不缩放不会报错，只是把分布推得更极端 —— 这是「静默的错」。")

## T6 为什么要除 sqrt(d_k)

In [ ]:
NK = 200
print("d_k    原始 std   缩放后 std   原始权重熵   缩放后权重熵   原始最大权重")
rows_dk = {}
for dk in (8, 64, 512):
    gg = torch.Generator().manual_seed(dk)
    q_ = torch.randn(1, 1, dk, generator=gg)
    k_ = torch.randn(1, NK, dk, generator=gg)

    raw_ = q_ @ k_.transpose(-1, -2)
    sc_ = raw_ / math.sqrt(dk)

    wr, ws = softmax_stable(raw_), softmax_stable(sc_)
    ent = lambda w: R(-(w * torch.log(w + 1e-12)).sum(-1).mean())  # noqa: E731
    rows_dk[dk] = dict(raw_std=R(raw_.std()), sc_std=R(sc_.std()),
                       raw_ent=ent(wr), sc_ent=ent(ws), raw_max=R(wr.max()))
    print("  %4d   %.6f    %.6f     %.6f     %.6f     %.6f"
          % (dk, rows_dk[dk]["raw_std"], rows_dk[dk]["sc_std"],
             rows_dk[dk]["raw_ent"], rows_dk[dk]["sc_ent"], rows_dk[dk]["raw_max"]))

print()
print("理论：scores 的 std ≈ sqrt(d_k) =",
      [round(math.sqrt(d), 3) for d in (8, 64, 512)])
print("d_k=512 时未缩放的 softmax 最大权重 %.6f —— 已经饱和成 one-hot，"
      % rows_dk[512]["raw_max"])
print("梯度会在 softmax 里消失。这就是那个 sqrt(d_k) 的全部意义。")

# ---- 验收 ----
assert rows_dk[8]["raw_max"] < rows_dk[64]["raw_max"] < rows_dk[512]["raw_max"]
assert rows_dk[512]["raw_max"] > 0.99, "d_k=512 时未缩放应当饱和"
assert rows_dk[512]["sc_ent"] > 4.0, "缩放之后分布应当重新变平"
print("\n注意：d_k 越大，问题越严重 —— 大模型动辄 d_k=64/128，这个缩放不是可选项。")

## T7 padding mask

In [ ]:
T_MASK = 5
pad_mask = torch.tensor([[False, False, False, True, True]])     # 后两位是 padding
s_mask = torch.randn(1, T_MASK, T_MASK, generator=torch.Generator().manual_seed(5))

w_pad = softmax_stable(s_mask.masked_fill(pad_mask.unsqueeze(1), float("-inf")))

pad_weight = R(w_pad[0, :, 3:].sum())

print("padding mask 后第 0 行:", [round(v, 6) for v in w_pad[0, 0].tolist()])
print("pad 列权重和 %.6f | 每行权重和 %.6f" % (pad_weight, R(w_pad.sum(-1).mean())))

# 对照：如果 mask 处填 0 而不是 -inf
w_zero = softmax_stable(s_mask.masked_fill(pad_mask.unsqueeze(1), 0.0))
print("若填 0 而不是 -inf，pad 列权重和 %.6f（照样分到注意力！）"
      % R(w_zero[0, :, 3:].sum()))

# ---- 验收 ----
assert pad_weight < 1e-9, f"pad 位置权重必须为 0，实际 {pad_weight}"
assert R(w_pad.sum(-1).mean()) > 0.999
assert R(w_zero[0, :, 3:].sum()) > 0.1, "填 0 的写法确实会泄漏"
print("\nmask 的语义是『在 softmax 之前把分数压到 -inf』，不是『softmax 之后置零』。")

## T8 causal mask

In [ ]:
causal = torch.triu(torch.ones(T_MASK, T_MASK, dtype=torch.bool), diagonal=1)

w_cau = softmax_stable(s_mask.masked_fill(causal.unsqueeze(0), float("-inf")))

future_weight = R(w_cau[0].triu(1).sum())

print("causal mask 上三角权重和: %.6f" % future_weight)
print("第 0 行:", [round(v, 6) for v in w_cau[0, 0].tolist()], "（只能看自己）")
print("第 2 行:", [round(v, 6) for v in w_cau[0, 2].tolist()], "（看前 3 个）")
print()
print("不做 causal mask 时，第 0 行能看到未来，权重 %.6f"
      % R(softmax_stable(s_mask)[0, 0, 1:].sum()))
print("→ 训练时看到答案，推理时没有答案，指标会突然崩掉")

# ---- 验收 ----
assert future_weight < 1e-9, "未来位置权重必须为 0"
assert near(R(w_cau[0, 0, 0]), 1.0, 1e-6), "第一个位置只能注意自己"
print("\n两种 mask 可以叠加：padding 管「哪些位置不该看」，causal 管「哪些还看不到」。")

## T9 多头 split-merge

In [ ]:
with torch.random.fork_rng():
    torch.manual_seed(22)
    mha = nn.MultiheadAttention(E_MHA, H, batch_first=True)


def mha_manual(x, mha, n_head):
    """手写多头：一次投影 -> 拆头 -> 各头独立注意力 -> 合并 -> 输出投影"""
    Bn, Tn, E_ = x.shape
    d_h = E_ // n_head
    qkv = x @ mha.in_proj_weight.T + mha.in_proj_bias
    q, k, v = qkv.chunk(3, dim=-1)

    q = q.reshape(Bn, Tn, n_head, d_h).transpose(1, 2)
    k = k.reshape(Bn, Tn, n_head, d_h).transpose(1, 2)
    v = v.reshape(Bn, Tn, n_head, d_h).transpose(1, 2)

    o, w = attention(q, k, v)

    o = o.transpose(1, 2).reshape(Bn, Tn, E_)
    return o @ mha.out_proj.weight.T + mha.out_proj.bias, w


# ---- 验收 ----
x_mha = torch.randn(2, 5, E_MHA, generator=torch.Generator().manual_seed(9))
o_my, w_my = mha_manual(x_mha, mha, H)
assert o_my.shape == (2, 5, E_MHA)
assert w_my.shape == (2, H, 5, 5), f"权重应当是每个头一份，实际 {tuple(w_my.shape)}"
print("手写多头输出形状", tuple(o_my.shape), "| 权重形状", tuple(w_my.shape))

## T10 与 `nn.MultiheadAttention` 对账

In [ ]:
with torch.no_grad():
    o_ref, w_ref = mha(x_mha, x_mha, x_mha)

diff_mha = R((o_my - o_ref).abs().max())

diff_w_avg = R((w_my.mean(1) - w_ref).abs().max())
diff_w_naive = R((w_my - w_ref).abs().max())

print("输出最大差            : %.3e" % diff_mha)
print("权重形状 手写", tuple(w_my.shape), "| 参考", tuple(w_ref.shape))
print("权重差（对 head 平均）: %.3e" % diff_w_avg)
print("权重差（忘记平均）    : %.3e   <- 差两个数量级，会让人以为写错了" % diff_w_naive)
print()
print("每个 head 行和 %.6f | 不同 head 权重差异 %.6f"
      % (R(w_my.sum(-1).mean()), R((w_my[0, 0] - w_my[0, 1]).abs().mean())))
n_in = mha.in_proj_weight.numel() + mha.in_proj_bias.numel()
n_out = mha.out_proj.weight.numel() + mha.out_proj.bias.numel()
print("参数量 in_proj %d + out_proj %d = %d（理论 4E^2+4E = %d）"
      % (n_in, n_out, n_in + n_out, 4 * E_MHA * E_MHA + 4 * E_MHA))

# ---- 验收 ----
assert diff_mha < 1e-6, f"手写多头与 nn.MultiheadAttention 不一致：{diff_mha}"
assert diff_w_avg < 1e-6, f"对 head 平均后应当对齐：{diff_w_avg}"
print("\n多头没有增加参数：维度被拆成 H 份，每份 d_head = E/H，总量不变。")

## T11 复杂度实测

In [ ]:
times = {}
for T_ in (64, 128, 256, 512, 1024):
    xx = torch.randn(8, T_, 64).requires_grad_(True)
    W_proj = torch.randn(64, 64)
    ts = []
    for _ in range(5):
        t0 = time.perf_counter()
        q_ = xx @ W_proj
        s_ = q_ @ q_.transpose(-2, -1) / 8.0
        a_ = softmax_stable(s_)
        o_ = a_ @ q_
        o_.sum().backward()
        ts.append(time.perf_counter() - t0)
        xx.grad = None
    ts.sort()
    times[T_] = ts[len(ts) // 2]
    print("T=%5d  前向+反向中位 %.6f s" % (T_, times[T_]))

print()
print("T 翻倍时的耗时比（理论 ×4，因为 scores 是 T×T）：")
ks = sorted(times)
for a_, b_ in zip(ks, ks[1:]):
    print("  %4d -> %4d : ×%.3f" % (a_, b_, times[b_] / times[a_]))

grow = times[1024] / times[256]
print("T ×4（256 -> 1024）耗时 ×%.2f（理论 ×16）" % grow)

# ---- 验收 ----
assert grow > 8, f"应当是平方级增长，实际只涨了 {grow} 倍"
print("\n这就是「长文本贵」的根源：注意力矩阵有 T^2 个元素。")

## T12 注意力矩阵元素数

In [ ]:
elems = {t_: t_ * t_ for t_ in (64, 256, 1024)}

for t_, n_ in elems.items():
    print("T=%5d  注意力矩阵元素数 %8d  （≈ %.2f MB float32）"
          % (t_, n_, n_ * 4 / 1024 / 1024))
print()
print("T 从 64 到 1024：元素数 ×%d" % (elems[1024] // elems[64]))
print("对比：序列本身的长度只涨了 16 倍。")

# ---- 验收 ----
assert elems[1024] == 1048576
assert elems[1024] / elems[256] == 16
print("\n→ 长上下文的所有工程手段（稀疏注意力 / 分块 / 线性注意力）都在打这个 T^2。")

## T13 置换等变性

In [ ]:
with torch.random.fork_rng():
    torch.manual_seed(33)
    lin_perm = nn.Linear(E_MHA, 3 * E_MHA)
W_p, b_p = lin_perm.weight.detach(), lin_perm.bias.detach()

x_perm_src = x_mha.clone()
perm = torch.tensor([3, 1, 4, 0, 2])
x_perm = x_perm_src[:, perm, :]


def attn_once(z):
    qkv = z @ W_p.T + b_p
    q_, k_, v_ = qkv.chunk(3, dim=-1)
    return attention(q_, k_, v_)


o_a, w_a = attn_once(x_perm_src)
o_b, w_b = attn_once(x_perm)

diff_perm_out = R((o_b - o_a[:, perm, :]).abs().max())

diff_perm_w = R((w_b - w_a[:, perm][:, :, perm]).abs().max())

print("置换输入后输出 vs 原输出置换 的最大差: %.3e" % diff_perm_out)
print("置换输入后权重 vs 原权重置换 的最大差: %.3e" % diff_perm_w)
print("mean pooling 对置换的最大差: %.3e"
      % R((x_perm.mean(1) - x_perm_src.mean(1)).abs().max()))

# ---- 验收 ----
assert diff_perm_out < 1e-6, "Attention 应当是置换等变的"
assert diff_perm_w < 1e-6
print("\n置换等变 = 换顺序，输出跟着换 —— 它没有『位置』这个概念。")
print("所以「渗油在前」和「渗油在后」对它是同一件事 —— 必须外挂位置编码。")

## T14 真实句子的注意力矩阵（分词工具已给，不挖空）

In [ ]:
# 最大匹配分词（ch01 已手写过，本章直接复用）
VOCAB_W = ["变压器", "断路器", "隔离开关", "避雷器", "绝缘子", "套管", "母线",
           "渗油", "漏油", "油迹", "滴油", "锈蚀", "生锈", "铁锈", "漆面剥落",
           "破损", "裂纹", "碎裂", "缺口", "掉瓷",
           "发热", "温度偏高", "过热", "烫手", "温升异常",
           "异物", "鸟巢", "塑料袋", "树枝", "风筝线搭挂",
           "巡检", "发现", "位于", "存在", "缺陷", "一般", "请", "尽快", "核实",
           "已", "通知", "运维班", "详见", "接报", "列入", "本周", "计划",
           "停电", "处理", "反馈", "安排", "抢修"]
VSET = set(VOCAB_W)
MAXL = max(len(w) for w in VOCAB_W)


def fmm(text, vocab=VSET, maxl=MAXL):
    """正向最大匹配；没命中的字符直接丢弃（标点、编号等在词表外）。"""
    out, i = [], 0
    while i < len(text):
        for L in range(min(maxl, len(text) - i), 0, -1):
            if text[i:i + L] in vocab:
                out.append(text[i:i + L])
                i += L
                break
        else:
            i += 1
    return out


print("词表", len(VOCAB_W), "个词 | 最长", MAXL, "字")

In [ ]:
CSV = Path("data") / "synth_text.csv"
df = pd.read_csv(CSV, keep_default_na=False)

row = df[df["缺陷类型"] == "渗漏油"].iloc[3]
toks = fmm(row["缺陷描述"])
print("工单编号", row["工单编号"], "| 缺陷类型", row["缺陷类型"])
print("原文:", row["缺陷描述"])
print("分词:", toks, "| token 数", len(toks))

In [ ]:
with torch.random.fork_rng():
    torch.manual_seed(44)
    emb = nn.Embedding(len(VOCAB_W) + 1, 16)      # 0 号留给「未知」
    lin_real = nn.Linear(16, 24)

ids = torch.tensor([[VOCAB_W.index(t) + 1 if t in VSET else 0 for t in toks]])
z_real = emb(ids)

qkv_r = z_real @ lin_real.weight.T + lin_real.bias
q_r, k_r, v_r = qkv_r.chunk(3, dim=-1)
out_r, w_real = attention(q_r, k_r, v_r)

print("注意力矩阵形状", tuple(w_real.shape), "| 每行和 %.6f" % R(w_real.sum(-1).mean()))
print("最大权重 %.6f | 最小权重 %.6f" % (R(w_real.max()), R(w_real.min())))
print("第 0 行权重:", [round(v, 4) for v in w_real[0, 0].tolist()])
print("上三角（未来位置）权重和 %.6f —— 没有 mask，它看得见全部"
      % R(w_real[0].triu(1).sum()))

# ---- 验收 ----
assert w_real.shape[1] == w_real.shape[2] == len(toks)
assert (w_real.sum(-1) - 1).abs().max() < 1e-6
print("\n注意：随机 embedding + 未训练，所以数值本身没有语义；")
print("但形状、归一化、全连接这三件事已经清清楚楚。")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.6, 4.0))

# 左：随机初始化下的真实句子
im0 = axes[0].imshow(w_real[0].detach().numpy(), cmap="viridis", vmin=0)
axes[0].set_xticks(range(len(toks)))
axes[0].set_yticks(range(len(toks)))
axes[0].set_xticklabels(toks, rotation=45, ha="right", fontsize=8)
axes[0].set_yticklabels(toks, fontsize=8)
axes[0].set_title("未训练：每个 token 都能看所有 token", fontsize=10)
fig.colorbar(im0, ax=axes[0], fraction=0.046)

# 右：causal mask 的效果（用同一批分数）
s_real = q_r @ k_r.transpose(-2, -1) / math.sqrt(q_r.shape[-1])
cm = torch.triu(torch.ones(len(toks), len(toks), dtype=torch.bool), diagonal=1)
w_causal_real = softmax_stable(s_real.masked_fill(cm.unsqueeze(0), float("-inf")))
im1 = axes[1].imshow(w_causal_real[0].detach().numpy(), cmap="viridis", vmin=0)
axes[1].set_xticks(range(len(toks)))
axes[1].set_yticks(range(len(toks)))
axes[1].set_xticklabels(toks, rotation=45, ha="right", fontsize=8)
axes[1].set_yticklabels(toks, fontsize=8)
axes[1].set_title("加 causal mask：只看得见左边", fontsize=10)
fig.colorbar(im1, ax=axes[1], fraction=0.046)

plt.tight_layout()
print("右上角（未来位置）全黑 —— mask 把上三角压成了 0。")

## 汇总

In [ ]:
import pandas as pd

rows = [
    ("稳定 softmax", "与 F.softmax 差 %.1e" % diff_softmax, "减 max 不改变结果，纯保险"),
    ("朴素 softmax 崩点", "z=100 出 nan", "溢出在 exp 里，不在除法里"),
    ("手写注意力 vs F.sdpa", "差 %.3e" % diff_sdpa, "缩放点积公式正确"),
    ("除 sqrt(d_k)", "d_k=512 未缩放最大权重 %.4f" % rows_dk[512]["raw_max"], "饱和成 one-hot"),
    ("padding mask", "pad 列权重和 %.1e" % pad_weight, "必须在 softmax 前填 -inf"),
    ("causal mask", "上三角权重和 %.1e" % future_weight, "否则训练时偷看答案"),
    ("多头 vs nn.MHA", "输出差 %.3e" % diff_mha, "拆头/合并顺序正确"),
    ("多头的权重对账", "平均后 %.1e / 忘记平均 %.2f" % (diff_w_avg, diff_w_naive),
     "参考实现默认已对 head 平均"),
    ("多头参数量", "%d = 4E^2+4E" % (n_in + n_out), "拆头不增加参数"),
    ("复杂度", "T×4 耗时 ×%.1f" % grow, "注意力矩阵 T^2 个元素"),
    ("置换等变", "差 %.1e" % diff_perm_out, "自带位置感 = 无，必须加编码"),
]
print(pd.DataFrame(rows, columns=["实验", "实测", "结论"]).to_string(index=False))

## 自查清单

- [ ] `softmax_stable` 与 `F.softmax` 差 **< 1e-9**，`softmax_naive` 在 `z=100` 出 **nan**
- [ ] Q / K / V 形状都是 `(2, 4, 8)`，且两种切法完全一致
- [ ] 手写注意力与 `F.sdpa` 差 **< 1e-6**（实际 1.19e-07）
- [ ] 每行注意力权重和 **= 1**
- [ ] `d_k=512` 时未缩放的最大权重 **> 0.99**（饱和）
- [ ] padding mask 后 pad 列权重和 **< 1e-9**；填 0 的写法权重和 **> 0.1**
- [ ] causal mask 后上三角权重和 **< 1e-9**，第 0 行只有自己 = 1
- [ ] 手写多头输出与 `nn.MultiheadAttention` 差 **< 1e-6**
- [ ] 权重对账**先对 head 求平均**（差 1e-8 而不是 0.19）
- [ ] 参数量 **= 4E²+4E**
- [ ] `T` 从 256 到 1024 耗时涨 **> 8 倍**
- [ ] 置换输入后输出差 **< 1e-6**（等变性成立）

**答不上来时问自己**：
1. 为什么减 max 不改变 softmax 的结果？
2. mask 为什么必须在 softmax **之前**生效？
3. `sqrt(d_k)` 是从哪推出来的？
4. `nn.MultiheadAttention` 的权重为什么比手写版少一个维度？
5. 「置换等变」和「置换不变」差在哪？